# DataHub Error Case 05 — Elasticsearch Error

**Monitor:** DataHub Elasticsearch Error  
**Component:** datahub-gms  
**Logger:** c.l.m.s.e.ESSearchDAO

In [1]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *
import urllib3
urllib3.disable_warnings()

r_login = get_token()
assert r_login.status_code == 200
session_cookie = r_login.cookies.get('actor')
headers = {'Cookie': f'actor={session_cookie}', 'Content-Type': 'application/json'}
print('Logged in')

✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
Logged in


In [8]:
# Cell 2: Check DataHub ES indices
r = requests.get(
    'https://192.168.194.1:9200/_cat/indices/datahub*?h=index,health,status,docs.count',
    auth=('admin', 'Str0ngP@ssw0rd123!'),
    verify=False
)
print('DataHub Elasticsearch indices:')
print(r.text)

DataHub Elasticsearch indices:



In [9]:
# Check all indices in OpenSearch
r = requests.get(
    'https://192.168.194.1:9200/_cat/indices?h=index,health,status,docs.count&s=index',
    auth=('admin', 'Str0ngP@ssw0rd123!'),
    verify=False
)
print(r.text)

.kibana_1                    green  open     39
.kibana_92668751_admin_1     green  open     32
.opendistro_security         green  open      9
.opensearch-observability    green  open      0
.plugins-ml-config           green  open      1
.ql-datasources              green  open      0
k8s-logs-2026.06.01          yellow open 808016
k8s-logs-2026.06.02          yellow open  75342
k8s-logs-2026.06.03          yellow open   7464
k8s-logs-2026.06.04          yellow open  39447
k8s-logs-2026.06.05          yellow open  10268
k8s-logs-2026.06.06          yellow open   1300
k8s-logs-2026.06.08          yellow open   7780
k8s-logs-2026.06.09          yellow open   5600
k8s-logs-2026.06.10          yellow open  20878
posts                        yellow open     21
profiles                     yellow open     12
security-auditlog-2026.04.07 green  open    136
security-auditlog-2026.04.08 green  open      1
security-auditlog-2026.06.01 green  open     39
security-auditlog-2026.06.02 green  open

In [ ]:
# Cell 3: Trigger search through DataHub GMS
query = '{ search(input: { type: DATASET, query: "test-nonexistent-xyz-12345", start: 0, count: 5 }) { total } }'
r2 = requests.post(
    f'{DATAHUB_URL}/api/graphql',
    headers=headers,
    json={'query': query}
)
print(f'Search: {r2.status_code}')
print(json.dumps(r2.json(), indent=2)[:400])

In [ ]:
# Cell 4: Find ES error logs
wait_for_logs(10)
client = os_client()
result = client.search(
    index=LOG_INDEX,
    body={
        'query': {
            'bool': {
                'must': [
                    {'range': {'@timestamp': {'gte': 'now-2m'}}},
                    {'term': {'kubernetes.container_name.keyword': 'datahub-gms'}},
                ],
                'should': [
                    {'match': {'message': 'Elasticsearch'}},
                    {'match': {'message': 'index_not_found'}},
                    {'match': {'message': 'SearchPhase'}},
                ],
                'minimum_should_match': 1,
            }
        },
        'sort': [{'@timestamp': {'order': 'desc'}}],
        'size': 10,
    }
)
print_logs(result['hits']['hits'])

In [ ]:
print('\n=== Summary ===')
print('Error:   Elasticsearch search/index error')
print('Monitor: DataHub Elasticsearch Error fires on count > 0 in 5 min')
print('Fix:     Re-run datahub-system-update if index missing')